In [ ]:
import os
import joblib
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

#  Define the absolute project path
BASE_DIR = '/content/drive/MyDrive/knn_project'

#  Load preprocessed dataset
X_train_scaled, X_test_scaled, y_train, y_test = joblib.load(
    os.path.join(BASE_DIR, 'data/processed/scaled_data.pkl')
)

#  Define the parameter grid space for Grid Search
param_grid = {
    'n_neighbors': list(range(1, 26, 2)),  # Odd values for k to prevent voting ties
    'weights': ['uniform', 'distance'],      # Vote weighting method
    'p': [1, 2]                              # p=1: Manhattan Distance | p=2: Euclidean Distance
}

#  Set up Grid Search with 5-Fold Cross-Validation
knn = KNeighborsClassifier()
grid_search = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

#  Fit Grid Search to training data
grid_search.fit(X_train_scaled, y_train)

#  Extract the best estimator
best_knn = grid_search.best_estimator_

#  Evaluate the best model on test data
y_pred = best_knn.predict(X_test_scaled)
test_acc = accuracy_score(y_test, y_pred)

print(" Hyperparameter Optimization Results:")
print(f"  Best Number of Neighbors (k): {grid_search.best_params_['n_neighbors']}")
print(f"  Weights: {grid_search.best_params_['weights']}")
print(f"  Distance Metric (p): {grid_search.best_params_['p']} ({'Manhattan' if grid_search.best_params_['p']==1 else 'Euclidean'})")
print(f" Cross-Validation Accuracy (CV Accuracy): {grid_search.best_score_ * 100:.2f}%")
print(f"  Final Test Accuracy: {test_acc * 100:.2f}%\n")

print(" Detailed Classification Report:")
print(classification_report(y_test, y_pred))

#  Save the best model to the models directory
model_path = os.path.join(BASE_DIR, 'models', 'knn_model.pkl')
joblib.dump(best_knn, model_path)
print(f" Final model saved successfully at: {model_path}")